# Decomposição do Projeto Final
## CallMeMaybe — Identificação de Operadores Ineficientes

| | |
|---|---|
| **Autor** | Raimir Silva |
| **Data** | Maio de 2026 |
| **Sprint** | 14 — Projeto Final |
| **Área** | Telecomunicações |

---

Este documento apresenta a decomposição estruturada das tarefas do projeto final. O objetivo é planejar, de forma clara e lógica, todas as etapas necessárias para resolver o problema de negócio antes de iniciar a implementação.

---

## 1. Identificação do Problema de Negócio

### Contexto Empresarial

A **CallMeMaybe** é um serviço de telefonia virtual cujos clientes são organizações que gerenciam grandes volumes de chamadas — tanto recebidas quanto realizadas por equipes de operadores. Os operadores também podem realizar chamadas internas entre si pela própria rede da plataforma.

A empresa está desenvolvendo uma nova funcionalidade que permitirá aos **supervisores identificar operadores menos eficientes** com base em dados objetivos de desempenho.

### Critérios de Ineficiência

Segundo a definição do negócio, um operador é considerado **ineficiente** quando apresenta ao menos uma das seguintes condições:

- Possui **alta taxa de chamadas recebidas perdidas** (internas ou externas);
- Apresenta **tempo de espera prolongado** nas chamadas recebidas;
- No caso de operadores responsáveis por chamadas de saída: realiza **poucas chamadas ativas**.

### Por que isso gera valor?

Em operações de telefonia, a ineficiência de operadores tem impacto direto em:

- **Satisfação do cliente final**: chamadas perdidas ou longos tempos de espera deterioram a experiência;
- **Custos operacionais**: operadores com baixo volume de saída representam subutilização de recursos;
- **Gestão de equipes**: sem métricas claras, supervisores tomam decisões subjetivas e inconsistentes.

A análise permitirá que a CallMeMaybe entregue aos seus clientes uma ferramenta de **monitoramento baseada em dados**, melhorando a qualidade do serviço e a tomada de decisão gerencial.

### Objetivo da Análise

> Desenvolver e validar um conjunto de métricas (KPIs) que permita classificar operadores como eficientes ou ineficientes, identificar os operadores que estão abaixo do padrão esperado e fornecer evidências estatísticas que sustentem essa classificação.

---

## 2. Descrição dos Dados

O projeto utiliza dois datasets fornecidos pela plataforma.

---

### 2.1 Dataset Principal — `telecom_dataset_us.csv`

Contém os registros de chamadas por operador, agregados por data.

| Coluna | Tipo Esperado | Descrição |
|---|---|---|
| `user_id` | int / str | ID da conta do cliente (organização) |
| `date` | datetime | Data em que as estatísticas foram coletadas |
| `direction` | str (categórico) | Direção da chamada: `'in'` (entrante) ou `'out'` (saída) |
| `internal` | bool | `True` se a chamada foi interna (entre operadores do mesmo cliente) |
| `operator_id` | int | Identificador único do operador |
| `is_missed_call` | bool | `True` se a chamada foi perdida |
| `calls_count` | int | Número de chamadas no período |
| `call_duration` | float | Duração da chamada em segundos (sem tempo de espera) |
| `total_call_duration` | float | Duração total da chamada (incluindo tempo de espera) |

---

### 2.2 Dataset de Clientes — `telecom_clients_us.csv`

Contém informações cadastrais dos clientes (organizações) que contratam o serviço.

| Coluna | Tipo Esperado | Descrição |
|---|---|---|
| `user_id` | int / str | ID do cliente — chave de junção com o dataset principal |
| `tariff_plan` | str | Plano tarifário atual contratado |
| `date_start` | datetime | Data de registro do cliente na plataforma |

---

### 2.3 Diagnóstico Inicial Planejado

Antes de iniciar as análises, será realizado um diagnóstico da qualidade dos dados cobrindo:

**Volume:**
- Número de linhas e colunas de cada dataset
- Período coberto pela coluna `date`
- Quantidade de operadores e clientes únicos

**Qualidade:**
- Percentual de valores ausentes por coluna
- Identificação de linhas duplicadas
- Verificação de tipos de variáveis (ex.: `date` como string ao invés de datetime)
- Valores inesperados em colunas categóricas (`direction`, `internal`, `is_missed_call`)
- Verificação de consistência: `call_duration` nunca deve ser maior que `total_call_duration`

**Resumo Estatístico:**
- Média, mediana, desvio padrão, mínimo e máximo de `calls_count`, `call_duration` e `total_call_duration`
- Distribuição de chamadas por direção (`in` / `out`) e tipo (interna / externa)
- Frequência de chamadas perdidas por operador e por cliente

---

### 2.4 Variáveis Mais Relevantes

| Variável | Papel na Análise |
|---|---|
| `operator_id` | Variável-chave de agregação — todas as métricas serão calculadas por operador |
| `is_missed_call` + `calls_count` | Base para calcular a taxa de chamadas perdidas |
| `total_call_duration` − `call_duration` | Tempo de espera por chamada (variável derivada) |
| `direction` | Distingue operadores de entrada (in) e saída (out) |
| `internal` | Permite separar chamadas internas das externas na análise de chamadas perdidas |
| `tariff_plan` | Contextualiza os clientes — pode revelar se o plano influencia o perfil de chamadas |

---

## 3. Hipóteses a Serem Validadas

As hipóteses abaixo foram elaboradas a partir dos critérios de ineficiência definidos pelo negócio e das variáveis disponíveis nos dados. Cada hipótese propõe uma relação mensurável que será verificada por métodos estatísticos.

---

### Hipótese 1 — Taxa de Chamadas Perdidas

> Operadores classificados como ineficientes apresentam uma taxa de chamadas perdidas significativamente maior do que a dos demais operadores.

| | |
|---|---|
| **H₀** | A taxa de chamadas perdidas de operadores ineficientes é igual à dos eficientes |
| **H₁** | A taxa de chamadas perdidas de operadores ineficientes é significativamente maior |
| **Método** | Teste Mann-Whitney U (dados provavelmente não-normais) |
| **Nível de significância** | α = 0,05 |

---

### Hipótese 2 — Tempo de Espera nas Chamadas Recebidas

> Operadores com baixo desempenho apresentam tempo médio de espera nas chamadas recebidas significativamente maior em comparação aos operadores eficientes.

| | |
|---|---|
| **H₀** | Não há diferença significativa no tempo médio de espera entre operadores eficientes e ineficientes |
| **H₁** | O tempo médio de espera dos operadores ineficientes é significativamente maior |
| **Método** | Teste Mann-Whitney U |
| **Nível de significância** | α = 0,05 |

---

### Hipótese 3 — Volume de Chamadas Ativas (Operadores de Saída)

> Operadores responsáveis por chamadas de saída e classificados como ineficientes realizam um volume de chamadas ativas significativamente menor do que os demais.

| | |
|---|---|
| **H₀** | O volume de chamadas ativas dos operadores ineficientes é igual ao dos eficientes |
| **H₁** | Os operadores ineficientes realizam significativamente menos chamadas ativas |
| **Método** | Teste Mann-Whitney U (aplicado apenas ao subconjunto `direction = 'out'`) |
| **Nível de significância** | α = 0,05 |

---

> **Nota metodológica:** A escolha do teste Mann-Whitney U em vez do teste t de Student se justifica pela ausência de garantia de distribuição normal dos dados — hipótese que será verificada durante a EDA com o teste de Shapiro-Wilk.

---

## 4. Indicadores-Chave (KPIs)

Os KPIs abaixo serão calculados **por operador**, agregando todos os seus registros disponíveis no dataset.

---

### KPI 1 — Taxa de Chamadas Perdidas (TCP)

Mede a proporção de chamadas recebidas que não foram atendidas pelo operador.

$$TCP = \frac{\sum calls\_count \text{ (onde } is\_missed\_call = True \text{)}}{\sum calls\_count \text{ (onde } direction = \text{'in'}\text{)}} \times 100$$

- **Unidade:** percentual (%)
- **Interpretação:** quanto maior, pior o desempenho do operador no atendimento

---

### KPI 2 — Tempo Médio de Espera (TME)

Mede, em média, quanto tempo os chamadores aguardam antes de ser atendidos pelo operador.

$$TME = \frac{\sum (total\_call\_duration - call\_duration)}{\sum calls\_count \text{ (onde } direction = \text{'in'}\text{)}}$$

- **Unidade:** segundos (s)
- **Interpretação:** quanto maior, mais tempo os clientes ficam esperando — sinal de ineficiência

---

### KPI 3 — Volume de Chamadas Ativas (VCA)

Mede o total de chamadas realizadas pelo operador (saída). Aplicado apenas a operadores com `direction = 'out'`.

$$VCA = \sum calls\_count \text{ (onde } direction = \text{'out'}\text{)}$$

- **Unidade:** número de chamadas
- **Interpretação:** operadores com VCA muito abaixo da mediana do grupo são considerados pouco produtivos

---

### KPI 4 — Índice de Eficiência do Operador (IEO)

Métrica composta para ranquear operadores em uma escala única de eficiência, combinando os três KPIs acima após normalização min-max.

$$IEO = w_1 \times (1 - TCP_{norm}) + w_2 \times (1 - TME_{norm}) + w_3 \times VCA_{norm}$$

onde:
- $TCP_{norm}$, $TME_{norm}$ e $VCA_{norm}$ são os KPIs normalizados entre 0 e 1 (min-max)
- Os pesos $w_1$, $w_2$, $w_3$ (com $w_1 + w_2 + w_3 = 1$) serão definidos durante a EDA com base na variabilidade observada em cada indicador

> O IEO será calculado separadamente para operadores de entrada e de saída, pois o VCA só se aplica ao segundo grupo.

---

## 5. Plano de Ação

O plano está dividido em três grupos de atividades. O projeto deve ser concluído em **duas semanas** (10 dias úteis).

---

### 5.1 Limpeza e Pré-processamento de Dados

| # | Atividade | Detalhamento | Tempo Estimado |
|---|---|---|---|
| 1 | **Carregamento e inspeção inicial** | Carregar os dois CSVs, verificar shape, tipos de colunas, primeiras linhas e `.info()` | 30 min |
| 2 | **Tratamento de valores ausentes** | Calcular % de nulos por coluna; decidir entre remoção, imputação ou manutenção com justificativa | 1 h |
| 3 | **Remoção de duplicatas** | Identificar e remover linhas duplicadas exatas; verificar duplicatas parciais por `operator_id` + `date` + `direction` | 30 min |
| 4 | **Conversão de tipos** | Converter `date` para datetime, garantir que `is_missed_call` e `internal` sejam booleanos, `operator_id` como int | 30 min |
| 5 | **Validação de consistência** | Verificar se `call_duration` ≤ `total_call_duration` em todos os registros; tratar anomalias encontradas | 30 min |
| 6 | **Junção dos datasets** | Unir `telecom_dataset` e `telecom_clients` via `user_id`; verificar registros sem correspondência | 30 min |
| 7 | **Criação de variável derivada** | Calcular `wait_time = total_call_duration − call_duration` para cada registro | 15 min |

---

### 5.2 Análise de Dados

| # | Atividade | Detalhamento | Tempo Estimado |
|---|---|---|---|
| 1 | **Análise Exploratória de Dados (EDA)** | Histogramas, boxplots e tabelas de frequência para todas as variáveis relevantes; distribuição por `direction`, `internal`, `tariff_plan` | 3 h |
| 2 | **Cálculo dos KPIs por operador** | Agregar por `operator_id`: TCP, TME e VCA conforme fórmulas definidas na seção 4 | 2 h |
| 3 | **Identificação de operadores ineficientes** | Usar o percentil 75 como limiar: operadores com TCP ou TME acima do P75, ou VCA abaixo do P25, serão sinalizados | 1 h |
| 4 | **Verificação de normalidade** | Aplicar teste de Shapiro-Wilk nos KPIs para justificar a escolha do teste estatístico | 30 min |
| 5 | **Teste de hipótese 1** | Mann-Whitney U: comparar TCP entre operadores eficientes e ineficientes | 45 min |
| 6 | **Teste de hipótese 2** | Mann-Whitney U: comparar TME entre os dois grupos | 45 min |
| 7 | **Teste de hipótese 3** | Mann-Whitney U: comparar VCA entre os dois grupos (subconjunto `direction = 'out'`) | 45 min |
| 8 | **Análise temporal** | Verificar evolução dos KPIs ao longo do tempo para identificar tendências ou sazonalidade | 2 h |

---

### 5.3 Comunicação de Resultados

| # | Atividade | Detalhamento | Tempo Estimado |
|---|---|---|---|
| 1 | **Dashboard interativo** | Construir painel com KPIs por operador, com filtros por data, direção e cliente; publicar no Tableau Public ou Looker Studio | 4 h |
| 2 | **Relatório final** | Documento no formato CAR (Challenge / Actions / Results) com os achados principais, impacto no negócio e recomendações práticas; entregar em PDF ou slides | 3 h |

---

## 6. Stakeholders Impactados

| Stakeholder | Área | Por que é impactado | Como os resultados serão usados |
|---|---|---|---|
| **Supervisores de call center** | Operações | Usuários diretos da funcionalidade — acompanharão o desempenho individual de cada operador | Tomada de decisão sobre coaching, realocação ou treinamentos corretivos |
| **Operadores** | Operações | Serão avaliados pelos KPIs; o resultado pode influenciar avaliações de desempenho | Feedback sobre pontos de melhoria e metas de produtividade |
| **Time de Produto (CallMeMaybe)** | Produto / TI | Responsável por integrar a metodologia desenvolvida na análise em uma funcionalidade do produto | Especificações técnicas para implementar o painel de operadores na plataforma |
| **Gestão / Diretoria dos clientes** | Estratégico | Tomará decisões estratégicas sobre dimensionamento de equipes e padrões mínimos de atendimento | Relatório executivo com visão consolidada da eficiência das equipes |

---

## 7. Cronograma Geral (2 Semanas)

| Semana | Dia(s) | Bloco | Atividades |
|---|---|---|---|
| **Semana 1** | Dias 1–2 | Pré-processamento | Carregamento, inspeção, limpeza, conversão de tipos, junção dos datasets, criação de variáveis derivadas |
| **Semana 1** | Dias 3–5 | EDA + KPIs | Análise exploratória completa, cálculo dos KPIs TCP, TME e VCA por operador, identificação inicial de ineficientes |
| **Semana 2** | Dias 6–8 | Testes estatísticos | Verificação de normalidade (Shapiro-Wilk), testes Mann-Whitney U para as três hipóteses, análise temporal dos KPIs |
| **Semana 2** | Dias 9–10 | Comunicação | Construção do dashboard, elaboração do relatório final (formato CAR), revisão geral e entrega |

---

> **Estimativa total de esforço:** ~22 horas de trabalho efetivo distribuídas ao longo de 10 dias úteis.